# 02 — Reward Model Exploration

Load the trained baseline reward model and run inference on 5 HH-RLHF test pairs.

> **Prerequisite:** run `python -m src.train_reward_model` from the repo root first.

In [ ]:
import sys
sys.path.insert(0, '..')  # make src/ importable from notebooks/

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

## Load model

In [ ]:
MODEL_PATH = "../results/reward_model_baseline"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()
print(f"Model loaded. Device: {next(model.parameters()).device}")

## Reward scoring helper

In [ ]:
def get_reward(text: str, max_length: int = 512) -> float:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    with torch.no_grad():
        logits = model(**inputs).logits
    return logits.squeeze().item()

## Load 5 test examples

In [ ]:
test_ds = load_dataset("Anthropic/hh-rlhf", split="test")
examples = [test_ds[i] for i in range(5)]

## Inference — chosen vs rejected reward scores

In [ ]:
results = []
for i, ex in enumerate(examples):
    chosen_score = get_reward(ex["chosen"])
    rejected_score = get_reward(ex["rejected"])
    correct = chosen_score > rejected_score
    results.append({
        "example": i + 1,
        "chosen_score": chosen_score,
        "rejected_score": rejected_score,
        "margin": chosen_score - rejected_score,
        "correct": correct,
    })
    print(f"Example {i+1}")
    print(f"  chosen   score: {chosen_score:+.4f}")
    print(f"  rejected score: {rejected_score:+.4f}")
    print(f"  margin:         {chosen_score - rejected_score:+.4f}")
    print(f"  correct order:  {correct}")
    print()

## Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\nAccuracy on 5 examples: {df['correct'].mean():.0%}")

## Inspect one pair in full

In [ ]:
idx = 0  # change to inspect a different example
ex = examples[idx]

print(f"{'='*60}")
print(f"Example {idx+1} — chosen  (score: {results[idx]['chosen_score']:+.4f})")
print(f"{'='*60}")
print(ex["chosen"])
print(f"\n{'='*60}")
print(f"Example {idx+1} — rejected (score: {results[idx]['rejected_score']:+.4f})")
print(f"{'='*60}")
print(ex["rejected"])